# TG1 Sousse - Analyse Exploratoire des Données

## Description du Dataset
Ce notebook analyse **TG1_Sousse_ML.csv** - données de **décharges partielles (PD)**.

**Caractéristiques:**
- **14,956 lignes** × **91 colonnes**
- Résolution: **Original**
- 4 canaux de mesure (CH1-CH4)

**Types de mesures:**
| Mesure | Description | Unité |
|--------|-------------|-------|
| CURRENT | Courant de décharge | µA |
| DISCHARGE_RATE | Taux de décharge | nC²/s |
| MAX_CHARGE | Charge maximale | nC |
| PULSE_COUNT | Nombre d'impulsions | count |

## 1. Import des Librairies

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

print('Librairies importées!')

## 2. Chargement des Données

In [ ]:
df = pd.read_csv('../LAST_DATA/TG1_Sousse_ML.csv')
print(f'Dataset chargé: {{df.shape[0]:,}} lignes, {{df.shape[1]}} colonnes')
df.head()

In [ ]:
df.info()

## 3. Structure par Canal

In [ ]:
channels = ['CH1', 'CH2', 'CH3', 'CH4']

print('Colonnes par canal:')
for ch in channels:
    ch_cols = [c for c in df.columns if c.startswith(ch)]
    print(f'  {ch}: {len(ch_cols)} variables')

## 4. Résumé Statistique

In [ ]:
current_cols = [f'{ch}_CURRENT_ABS_uA' for ch in channels]
print('STATISTIQUES - COURANTS')
df[current_cols].describe()

## 5. Valeurs Manquantes

In [ ]:
missing = df.isnull().sum().sum()
print(f'Total valeurs manquantes: {missing}')

## 6. Distribution des Courants

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
colors = ['#2196F3', '#F44336', '#4CAF50', '#9C27B0']

for i, ch in enumerate(channels):
    col = f'{ch}_CURRENT_ABS_uA'
    sns.histplot(df[col], kde=True, ax=axes[i], color=colors[i], alpha=0.7, bins=50)
    axes[i].set_title(f'{ch} - Courant (µA)', fontsize=12, fontweight='bold')
    axes[i].axvline(df[col].mean(), color='red', linestyle='--')

plt.suptitle('Distribution des Courants de Décharge', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution des taux de décharge
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, ch in enumerate(channels):
    col = f'{ch}_DISCHARGE_RATE_ABS_nC2_per_s'
    data = df[col][df[col] > 0]
    sns.histplot(data, kde=True, ax=axes[i], color=colors[i], alpha=0.7, bins=50)
    axes[i].set_title(f'{ch} - Taux de Décharge', fontsize=12, fontweight='bold')

plt.suptitle('Distribution des Taux de Décharge', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Corrélation entre Canaux

In [ ]:
current_cols = [f'{ch}_CURRENT_ABS_uA' for ch in channels]
corr = df[current_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdYlBu_r', center=0, square=True)
plt.title('Corrélation entre Canaux', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Évolution Temporelle

In [ ]:
if all(col in df.columns for col in ['Year', 'Month', 'Day', 'Hour', 'Minute']):
    df['DateTime'] = pd.to_datetime(df[['Year', 'Month', 'Day', 'Hour', 'Minute']])
    
    step = max(1, len(df) // 2000)
    sample = df.iloc[::step].copy()
    
    fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)
    
    for i, ch in enumerate(channels):
        col = f'{ch}_CURRENT_ABS_uA'
        axes[i].plot(sample['DateTime'], sample[col], color=colors[i], linewidth=0.3)
        axes[i].fill_between(sample['DateTime'], sample[col], alpha=0.2, color=colors[i])
        axes[i].set_ylabel(f'{ch} (µA)')
        axes[i].set_title(ch)
    
    axes[-1].set_xlabel('Date')
    plt.suptitle('Évolution Temporelle', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# Analyse par heure
if 'Hour' in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    for i, ch in enumerate(channels):
        col = f'{ch}_CURRENT_ABS_uA'
        hourly = df.groupby('Hour')[col].mean()
        axes[i].bar(hourly.index, hourly.values, color=colors[i], edgecolor='black')
        axes[i].set_xlabel('Heure')
        axes[i].set_ylabel('Courant Moyen (µA)')
        axes[i].set_title(ch)
    
    plt.suptitle('Courant par Heure', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

## 9. Box Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Courant ABS
current_abs = [f'{ch}_CURRENT_ABS_uA' for ch in channels]
df[current_abs].boxplot(ax=axes[0])
axes[0].set_title('Courant Absolu')
axes[0].set_xticklabels(channels)

# Courant NEG
current_neg = [f'{ch}_CURRENT_NEG_uA' for ch in channels]
df[current_neg].boxplot(ax=axes[1])
axes[1].set_title('Courant Négatif')
axes[1].set_xticklabels(channels)

# Courant POS
current_pos = [f'{ch}_CURRENT_POS_uA' for ch in channels]
df[current_pos].boxplot(ax=axes[2])
axes[2].set_title('Courant Positif')
axes[2].set_xticklabels(channels)

plt.suptitle('Détection des Outliers', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Violin plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, ch in enumerate(channels):
    data = pd.DataFrame({
        'ABS': df[f'{ch}_CURRENT_ABS_uA'],
        'NEG': df[f'{ch}_CURRENT_NEG_uA'],
        'POS': df[f'{ch}_CURRENT_POS_uA']
    }).melt(var_name='Type', value_name='Courant')
    
    sns.violinplot(data=data, x='Type', y='Courant', ax=axes[i], palette='Set2')
    axes[i].set_title(ch)

plt.suptitle('Distribution des Courants', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 10. Relations entre Variables

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

sample = df.sample(n=min(5000, len(df)), random_state=42)

for i, ch in enumerate(channels):
    x = sample[f'{ch}_CURRENT_ABS_uA']
    y = sample[f'{ch}_DISCHARGE_RATE_ABS_nC2_per_s']
    
    axes[i].scatter(x, y, alpha=0.3, s=5, c=colors[i])
    axes[i].set_xlabel('Courant (µA)')
    axes[i].set_ylabel('Taux de Décharge (nC²/s)')
    axes[i].set_title(ch)

plt.suptitle('Courant vs Taux de Décharge', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Comparaison entre canaux
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# CH1 vs CH2
axes[0].scatter(sample['CH1_CURRENT_ABS_uA'], sample['CH2_CURRENT_ABS_uA'], alpha=0.3, s=5)
axes[0].set_xlabel('CH1 (µA)')
axes[0].set_ylabel('CH2 (µA)')
axes[0].set_title('CH1 vs CH2')

# CH3 vs CH4
axes[1].scatter(sample['CH3_CURRENT_ABS_uA'], sample['CH4_CURRENT_ABS_uA'], alpha=0.3, s=5, c='green')
axes[1].set_xlabel('CH3 (µA)')
axes[1].set_ylabel('CH4 (µA)')
axes[1].set_title('CH3 vs CH4')

# Moyennes
means = [df[f'{ch}_CURRENT_ABS_uA'].mean() for ch in channels]
axes[2].bar(channels, means, color=colors, edgecolor='black')
axes[2].set_ylabel('Courant Moyen (µA)')
axes[2].set_title('Comparaison Moyennes')

plt.tight_layout()
plt.show()

## 11. Conclusions

### Résumé:
- **Données de décharges partielles** sur 4 canaux
- **Pas de valeurs manquantes**
- Distribution asymétrique (beaucoup de valeurs faibles)
- Forte corrélation courant-taux de décharge
- Utile pour maintenance prédictive et détection d'anomalies